In [1]:
# Install required package
!pip install nba_api

import json
from nba_api.stats.endpoints import teamestimatedmetrics
from nba_api.stats.endpoints import TeamVsPlayer
from nba_api.stats.endpoints import PlayerGameLogs
from nba_api.stats.endpoints import playergamelog
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.static import teams
from nba_api.stats.endpoints import commonteamroster
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display
import time  # Ensure time is imported
from bs4 import BeautifulSoup

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # Set timezone to EST
        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch player positions from RealGM
def fetch_player_positions():
    url = "https://basketball.realgm.com/nba/players"
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")

        table = soup.find("table", {"class": "tablesaw"})
        player_data = {}

        if table:
            rows = table.find("tbody").find_all("tr")
            for row in rows:
                cols = row.find_all("td")
                if len(cols) >= 3:  # Ensure there are enough columns
                    player = cols[1].text.strip()  # Index 1 for Player
                    pos = cols[2].text.strip()  # Index 2 for Pos
                    player_data[player] = pos  # Store in dictionary {Player: Position}
        else:
            print("Table not found on the webpage.")
        return player_data
    except Exception as e:
        print(f"Error fetching player positions: {e}")
        return {}

# Fetch team metrics
def fetch_team_metrics():
    try:
        team_metrics = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        )
        data = team_metrics.get_data_frames()[0]
        metrics = data[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        # Calculate league averages
        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        return metrics_dict, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Adjust predictions
def adjust_prediction(stat_mean, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating):
    try:
        pace_factor = team_metrics.get('E_PACE', avg_pace) / avg_pace if avg_pace != 0 else 1
        offensive_factor = team_metrics.get('E_OFF_RATING', avg_off_rating) / avg_off_rating if avg_off_rating != 0 else 1
        defensive_factor = opposing_metrics.get('E_DEF_RATING', avg_def_rating) / avg_def_rating if avg_def_rating != 0 else 1
        adjusted_stat = stat_mean * pace_factor * offensive_factor / defensive_factor
        return adjusted_stat
    except Exception as e:
        print(f"Error adjusting prediction: {e}")
        return stat_mean

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        logs = playergamelog.PlayerGameLog(player_id=player_id, season="2024-25", season_type_all_star="Regular Season")
        df = logs.get_data_frames()[0]
        return df
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()

# Fetch statistics
def calculate_stats(series):
    if series.empty:
        return 0, 0, 0
    mean = series.mean()
    std_dev = series.std()
    cv = (std_dev / mean) if mean != 0 else 0
    return mean, std_dev, cv

# Analyze teams and players for today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_positions = fetch_player_positions()

    if not games_today:
        print("No games scheduled for today.")
        return pd.DataFrame()

    all_players_data = []

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        if home_team not in team_metrics or away_team not in team_metrics:
            print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
            continue

        for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
            try:
                team_roster = commonteamroster.CommonTeamRoster(
                    team_id=teams.find_team_by_abbreviation(team_name)['id']
                ).get_data_frames()[0]

                for _, player in team_roster.iterrows():
                    player_id = player['PLAYER_ID']
                    player_name = player['PLAYER']
                    print(f"Fetching data for {player_name} ({team_name})")

                    game_logs = fetch_player_game_logs(player_id)
                    if game_logs.empty:
                        continue

                    mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                    mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                    mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                    adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)
                    adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)
                    adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)

                    player_pos = player_positions.get(player_name, 'Unknown')

                    all_players_data.append({
                        'Player': player_name,
                        'Position': player_pos,
                        'Team': team_name,
                        'PPG': round(mean_ppg, 2),
                        'Adj PPG': round(adjusted_ppg, 2),
                        'PPG SD': round(std_dev_ppg, 2),
                        'PPG CV': round(cv_ppg, 2),
                        'RPG': round(mean_rpg, 2),
                        'Adj RPG': round(adjusted_rpg, 2),
                        'RPG SD': round(std_dev_rpg, 2),
                        'RPG CV': round(cv_rpg, 2),
                        'APG': round(mean_apg, 2),
                        'Adj APG': round(adjusted_apg, 2),
                        'APG SD': round(std_dev_apg, 2),
                        'APG CV': round(cv_apg, 2)
                    })
                    time.sleep(1)  # Respect API rate limits
            except Exception as e:
                print(f"Error analyzing team {team_name} or player {player_name}: {e}")
                continue

    if all_players_data:
        df = pd.DataFrame(all_players_data)
        display(df)
        return df
    else:
        return pd.DataFrame()

# Start the analysis
analyze_teams_for_today()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 5.0 MB/s eta 0:00:00
Metrics not available for one or both teams: CHA, WAS. Skipping game.
Metrics not available for one or both teams: DET, ATL. Skipping game.
Metrics not available for one or both teams: NYK, HOU. Skipping game.
Metrics not available for one or both teams: MEM, SAS. Skipping game.
Metrics not available for one or both teams: MIN, SAC. Skipping game.
Metrics not available for one or both teams: OKC, MIL. Skipping game.
Metrics not available for one or both teams: DEN, NOP. Skipping game.
Metrics not available for one or both teams: UTA, IND. Skipping game.
Metrics not available for one or both teams: GSW, ORL. Skipping game.
Metrics not available for one or both teams: POR, PHX. Skipping game.


""
